In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
import time


In [0]:
from pyspark.sql import SparkSession


# motivation

- batch-processing can suffer from `back-pressure` problem as the data increases and we are doing full load and full processing.<br>
- To overcome `back-pressure` it we can use `incremental processing` using `spark structure streaming` this will elemenate the full load and full processing but in this Case only new data is processed.

In [0]:
%run /Users/vivekkumarverma332@gmail.com/creds/creds

In [0]:
creds=get_creds()

In [0]:
base_dir=creds.get('project_base_dir', None)


In [0]:
dataset_dir=creds.get("dataset_dir",None)

In [0]:
creds

Out[21]: {'dataset_dir': '/FileStore/datasets/',
 'project_base_dir': '/FileStore/test_project/'}

In [0]:
project_name="test_proj_1"

In [0]:
dbutils.fs.ls(dataset_dir+"/invoices/")

Out[104]: [FileInfo(path='dbfs:/FileStore/datasets/invoices/invoices_1.json', name='invoices_1.json', size=354143, modificationTime=1736395535000),
 FileInfo(path='dbfs:/FileStore/datasets/invoices/invoices_2.json', name='invoices_2.json', size=344396, modificationTime=1736395535000),
 FileInfo(path='dbfs:/FileStore/datasets/invoices/invoices_3.json', name='invoices_3.json', size=412765, modificationTime=1736395536000),
 FileInfo(path='dbfs:/FileStore/datasets/invoices/invoices_4.json', name='invoices_4.json', size=7230, modificationTime=1736395536000)]

In [0]:
landing_place=f"{base_dir}{project_name}/landing_zone"
archieve_place=f"{base_dir}{project_name}/archieve_zone"
checkpoint_zone=f"{base_dir}{project_name}/checkpoint_zone"

In [0]:
print(dbutils.fs.help())

dbutils.fs provides utilities for working with FileSystems. Most methods in
this package can take either a DBFS path (e.g., "/foo" or "dbfs:/foo"), or
another FileSystem URI.

For more info about a method, use dbutils.fs.help("methodName") .

In notebooks, you can also use the %fs shorthand to access DBFS. The %fs shorthand maps
straightforwardly onto dbutils calls. For example, "%fs head --maxBytes=10000 /file/path"
translates into "dbutils.fs.head("/file/path", maxBytes = 10000)".
 fsutils cp(from: String, to: String, recurse: boolean = false): boolean -> Copies a file or directory, possibly across FileSystems head(file: String, maxBytes: int = 65536): String -> Returns up to the first 'maxBytes' bytes of the given file as a String encoded in UTF-8 ls(dir: String): Seq -> Lists the contents of a directory mkdirs(dir: String): boolean -> Creates the given directory if it does not exist, also creating any necessary parent directories mv(from: String, to: String, recurse: boolean = false): boolean -> Moves a file or directory, possibly across FileSystems put(file: String, contents: String, overwrite: boolean = false): boolean -> Writes the given String out to a file, encoded in UTF-8 rm(dir: String, recurse: boolean = false): boolean -> Removes a file or directory mount mount(source: String, mountPoint: String, encryptionType: String = "", owner: String = null, extraConfigs: Map = Map.empty[String, String]): boolean -> Mounts the given source directory into DBFS at the given mount point mounts: Seq -> Displays information about what is mounted within DBFS refreshMounts: boolean -> Forces all machines in this cluster to refresh their mount cache, ensuring they receive the most recent information unmount(mountPoint: String): boolean -> Deletes a DBFS mount point updateMount(source: String, mountPoint: String, encryptionType: String = "", owner: String = null, extraConfigs: Map = Map.empty[String, String]): boolean -> Similar to mount(), but updates an existing mount point (if present) instead of creating a new one

None


In [0]:
class TestProj1:
    # constructor of TestProj1
    def __init__(self,project_name:str,spark:SparkSession, landing_zone_dir, check_point_dir, archieve_dir:str=None)->None:
        self.landing_dir=landing_zone_dir
        self.checkpoint_dir=check_point_dir
        self.archieve_dir=archieve_dir
        self.spark:SparkSession=spark
        self.project_name=project_name
        self.schema=None
        self.bronzeLayerTableName=project_name+"_bronzeLayerTable"
        self.bronzeLayerStreamingQuery=None
        self.silverLayerTableName=project_name+"_silverLayerTable"
        self.silverLayerStreamingQuery=None
        self.goldLayerTableName=project_name+"_goldLayerTable"
        self.goldLayerStreamingQuery=None
        self.data=None
        self.sourceScheme= """
                    InvoiceNo string, StockCode string, Description string, Quantity integer, InvoiceDate string, UnitProce float, CustomerID integer, Country string
                """

    def cleaningAndSetup(self):
        dbutils.fs.rm(self.archieve_dir, True)
        dbutils.fs.rm(self.landing_dir, True)
        dbutils.fs.rm(self.checkpoint_dir, True)

        spark.sql(f"drop table if exists {self.bronzeLayerTableName}")
        spark.sql(f"drop table if exists {self.silverLayerTableName}")
        spark.sql(f"drop table if exists {self.goldLayerTableName}")

        dbutils.fs.rm(f"/user/hive/warehouse/{self.bronzeLayerTableName}", True)
        dbutils.fs.rm(f"/user/hive/warehouse/{self.silverLayerTableName}", True)
        dbutils.fs.rm(f"/user/hive/warehouse/{self.goldLayerTableName}", True)

        dbutils.fs.mkdirs(self.archieve_dir)
        dbutils.fs.mkdirs(self.landing_dir)

    def copy_to_landing_zone(self, file_dir, recursive:bool=False)->None:
        """copy file from file_dir to project landing zone"""
        res=dbutils.fs.cp(file_dir, self.landing_dir, recursive)
        if res:
            if recursive:
                print(f"FILES HAVE BEEN SUCCESSFULLY COPIED FROM {file_dir} to {self.landing_dir}")
            else:
                print(f"[{file_dir}] HAS BEEN SUCCESSFULLY COPIED TO {self.landing_dir}")
    
    def set_schema(self, schema):
        """sets the schema for data read"""
        self.schema=schema

    def bronze_layer(self, fileDir:str, fileFormat:str="CSV"):
        """[BRONZE LAYER] ingests data from landing zone and inserts it in project's broze table"""
        if fileFormat=="CSV":
            df=self.spark\
                .readStream\
                    .format("CSV")\
                        .schema(self.schema)\
                            .option("header", True)\
                                .option("cleanSource","archive")\
                                    .option("sourceArchiveDir", self.archieve_dir+"/bronze")\
                                        .load(fileDir)
        elif fileFormat=="JSON":
            df=self.spark\
                .readStream\
                    .format("JSON")\
                        .schema(self.schema)\
                            .option("cleanSource","archive")\
                                .option("sourceArchiveDir", self.archieve_dir+"/bronze")\
                                    .load(fileDir)
        # process and perform transformations

        print("[BRONZE LAYER TRANSFORMATION] waiting for 10 sec")
        time.sleep(10)
        # write the data to the bronze Layer cluster
        bronzeTableIngestionStreamQuery=df.writeStream\
            .format('delta')\
                .queryName("bronzeLayerIngestionSQ")\
                    .option("checkpointLocation", self.checkpoint_dir+'/bronze')\
                        .option("maxFilesPerTrigger", 1)\
                            .trigger(processingTime ='1 minute')\
                                .outputMode("append")\
                                    .toTable(self.bronzeLayerTableName)
        self.bronzeLayerStreamingQuery=bronzeTableIngestionStreamQuery
        return bronzeTableIngestionStreamQuery
    
    def silver_layer(self):
        """[SILVER LAYER] data read from bronze layer and data cleaning and save to silver layer"""
        df=self.spark\
            .readStream\
                .format("delta")\
                    .table(self.bronzeLayerTableName)
        # process and perform transformations

        print("[SILVER LAYER TRANSFORMATION] waiting for 10 sec")
        time.sleep(10)

        # write the data to the silver Layer cluster
        silverLayerStreamQuery=df.writeStream\
            .format("delta")\
                .queryName("SilverLayerSQ")\
                    .option("checkpointLocation", self.checkpoint_dir+'/silver')\
                        .trigger(processingTime='1 minute')\
                            .outputMode("append")\
                                .toTable(self.silverLayerTableName)

        self.silverLayerStreamingQuery=silverLayerStreamQuery
        return silverLayerStreamQuery
    
    def gold_layer(self):
        """[GOLD LAYER] process the data at the silver layer table and save it in a gold layer table"""
        df=self.spark\
            .readStream\
                .format("delta")\
                    .table(self.silverLayerTableName)


        # process and perform transformations

        print("[GOLD LAYER TRANSFORMATION] waiting for 10 sec")
        time.sleep(10)
        # write the data to the gold Layer cluster
        goldLayerStreamingQuery=df.writeStream\
            .format('delta')\
                .queryName("GoldLayerSQ")\
                    .option("checkpointLocation",self.checkpoint_dir+"/gold")\
                        .outputMode("append")\
                            .trigger(processingTime="1 minute")\
                                .toTable(self.goldLayerTableName)
        self.goldLayerStreamingQuery=goldLayerStreamingQuery

    def stopBronzeLayerSQ(self):
        res=self.bronzeLayerStreamingQuery.awaitTermination(10000) #waiting for 10000 ms
        return res
    
    def stopSilverLayerSQ(self):
        res=self.silverLayerStreamingQuery.awaitTermination(10000) #waiting for 10000 ms
        return res
    
    def stopGoldLayerSQ(self):
        res=self.goldLayerStreamingQuery.awaitTermination(10000) #waiting for 10000 ms
        return res


    def start_process(self, fileFormat:str="CSV"):
        self.bronze_layer(self.landing_dir, fileFormat=fileFormat)
        print("BRONZE LAYER STREAMING STARTED !")
        self.silver_layer()
        print("SILVER LAYER STREAMING STARTED !")
        self.gold_layer()
        print("GOLD LAYER STREAMING STARTED !")

    def stop_all_process(self):
        self.stopBronzeLayerSQ()
        print("BRONZE LAYER STOPPED !")
        self.stopSilverLayerSQ()
        print("SILVER LAYER STREAMING STOPPED !")
        self.stopGoldLayerSQ()
        print("GOLD LAYER STREAMING STOPPED !")

    


In [0]:
MDL_ARCH_DEMO=TestProj1(
    project_name=project_name, 
    spark=spark, 
    landing_zone_dir=landing_place, 
    check_point_dir=checkpoint_zone,
    archieve_dir=archieve_place
)


In [0]:
MDL_ARCH_DEMO.cleaningAndSetup()

In [0]:
invoice_schema_string = """
CESS DOUBLE,
CGST DOUBLE,
CashierID STRING,
CreatedTime LONG,
CustomerCardNo STRING,
CustomerType STRING,
DeliveryAddress STRUCT<
    AddressLine: STRING,
    City: STRING,
    ContactNumber: STRING,
    PinCode: STRING,
    State: STRING
>,
DeliveryType STRING,
InvoiceLineItems ARRAY<STRUCT<
    ItemCode: STRING,
    ItemDescription: STRING,
    ItemPrice: DOUBLE,
    ItemQty: LONG,
    TotalValue: DOUBLE
>>,
InvoiceNumber STRING,
NumberOfItems LONG,
PaymentMethod STRING,
PosID STRING,
SGST DOUBLE,
StoreID STRING,
TaxableAmount DOUBLE,
TotalAmount DOUBLE
"""


In [0]:
schema_string = "TransactionID INT, Date STRING, Product STRING, Category STRING, Quantity INT, PricePerUnit DOUBLE, TotalAmount DOUBLE, CustomerID INT"


In [0]:
MDL_ARCH_DEMO.set_schema(schema_string)

In [0]:
MDL_ARCH_DEMO.start_process(fileFormat="CSV")

[BRONZE LAYER TRANSFORMATION] waiting for 10 sec
BRONZE LAYER STREAMING STARTED !
[SILVER LAYER TRANSFORMATION] waiting for 10 sec
SILVER LAYER STREAMING STARTED !
[GOLD LAYER TRANSFORMATION] waiting for 10 sec
GOLD LAYER STREAMING STARTED !


In [0]:
for i in range(1,3,1):
    MDL_ARCH_DEMO.copy_to_landing_zone(f'dbfs:/FileStore/datasets/sales_data/sales_data_csv{i}.csv')
    time.sleep(50)


[dbfs:/FileStore/datasets/sales_data/sales_data_csv1.csv] HAS BEEN SUCCESSFULLY COPIED TO /FileStore/test_project/test_proj_1/landing_zone
[dbfs:/FileStore/datasets/sales_data/sales_data_csv2.csv] HAS BEEN SUCCESSFULLY COPIED TO /FileStore/test_project/test_proj_1/landing_zone


In [0]:
MDL_ARCH_DEMO.stop_all_process()

BRONZE LAYER STOPPED !
SILVER LAYER STREAMING STOPPED !
GOLD LAYER STREAMING STOPPED !


structure streaming offers `three trigger type`:
1.  `Unspecified`
2.  `processingTime`
3.  `availableNow`

In [0]:
dbutils.fs.ls("dbfs:/databricks-datasets/online_retail/data-001/data.csv")

Out[9]: [FileInfo(path='dbfs:/databricks-datasets/online_retail/data-001/data.csv', name='data.csv', size=5357240, modificationTime=1466107812000)]

In [0]:
dbutils.fs.ls("dbfs:/")

Out[13]: [FileInfo(path='dbfs:/FileStore/', name='FileStore/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/databricks-datasets/', name='databricks-datasets/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/databricks-results/', name='databricks-results/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/mnt/', name='mnt/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/user/', name='user/', size=0, modificationTime=0)]

In [0]:
%fs ls dbfs:/

path,name,size,modificationTime
dbfs:/FileStore/,FileStore/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0
dbfs:/databricks-results/,databricks-results/,0,0
dbfs:/mnt/,mnt/,0,0
dbfs:/user/,user/,0,0


In [0]:
%fs mounts

mountPoint,source,encryptionType
/databricks-datasets,databricks-datasets,
/databricks/mlflow-tracking,databricks/mlflow-tracking,sse-s3
/databricks-results,databricks-results,sse-s3
/databricks/mlflow-registry,databricks/mlflow-registry,sse-s3
/,DatabricksRoot,sse-s3


In [0]:
spark.sparkContext

SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

In [0]:
bucket_name

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-3789068592952830>:1
----> 1 bucket_name

NameError: name 'bucket_name' is not defined

In [0]:
filedir="dbfs:/databricks-datasets/online_retail/data-001/"

In [0]:
spark.conf.set("spark.app.name", project_name)

In [0]:
spark.sparkContext

SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

In [0]:
spark.sparkContext._conf.get("spark.app.name")

Out[20]: 'Databricks Shell'

In [0]:
df=spark.read.format("CSV").option("inferSchema","true").option("header",True).load("dbfs:/databricks-datasets/online_retail/data-001/data.csv")

In [0]:
df.columns

Out[32]: ['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

In [0]:
display(df.head(10))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/10 8:26,7.65,17850,United Kingdom
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/10 8:26,4.25,17850,United Kingdom
536366,22633,HAND WARMER UNION JACK,6,12/1/10 8:28,1.85,17850,United Kingdom
536366,22632,HAND WARMER RED POLKA DOT,6,12/1/10 8:28,1.85,17850,United Kingdom
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/10 8:34,1.69,13047,United Kingdom


In [0]:
df.schema

Out[34]: StructType([StructField('InvoiceNo', StringType(), True), StructField('StockCode', StringType(), True), StructField('Description', StringType(), True), StructField('Quantity', IntegerType(), True), StructField('InvoiceDate', StringType(), True), StructField('UnitPrice', DoubleType(), True), StructField('CustomerID', IntegerType(), True), StructField('Country', StringType(), True)])

In [0]:
schema=StructType([StructField('InvoiceNo', StringType(), True), StructField('StockCode', StringType(), True), StructField('Description', StringType(), True), StructField('Quantity', IntegerType(), True), StructField('InvoiceDate', StringType(), True), StructField('UnitPrice', DoubleType(), True), StructField('CustomerID', IntegerType(), True), StructField('Country', StringType(), True)])

In [0]:
schema= """
    InvoiceNo string, StockCode string, Description string, Quantity integer, InvoiceDate string, UnitProce float, CustomerID integer, Country string
"""

In [0]:
df_test=spark.read.format("JSON").option("inferSchema", True).load("dbfs:/FileStore/datasets/invoices/invoices_1.json")

In [0]:
df_test.schema

Out[89]: StructType([StructField('CESS', DoubleType(), True), StructField('CGST', DoubleType(), True), StructField('CashierID', StringType(), True), StructField('CreatedTime', LongType(), True), StructField('CustomerCardNo', StringType(), True), StructField('CustomerType', StringType(), True), StructField('DeliveryAddress', StructType([StructField('AddressLine', StringType(), True), StructField('City', StringType(), True), StructField('ContactNumber', StringType(), True), StructField('PinCode', StringType(), True), StructField('State', StringType(), True)]), True), StructField('DeliveryType', StringType(), True), StructField('InvoiceLineItems', ArrayType(StructType([StructField('ItemCode', StringType(), True), StructField('ItemDescription', StringType(), True), StructField('ItemPrice', DoubleType(), True), StructField('ItemQty', LongType(), True), StructField('TotalValue', DoubleType(), True)]), True), True), StructField('InvoiceNumber', StringType(), True), StructField('NumberOfItems'